# Phase A1 — Kong-protocol subset eval (Table A1 appendix)

**Goal:** Direct numerical comparison vs Kong et al. 2025 Table 1 dùng
Kong-protocol matched subset (~907 random Fold 3 patches, ~33% per Kong's 5:1:3 split).

**Khác A1 extended full Fold 3:**
- N = 907 random patches (vs 2722 full)
- Seed cố định cho reproducibility
- Output cho Table A1 appendix paper

**Compute:** ~30 min trên T4.

## 00 — Setup

In [ ]:
import subprocess, sys, os, platform, time, json
print("Python:", sys.version.split()[0])
import torch
print("Torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

In [ ]:
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"
DATA_ROOT = "/kaggle/input/datasets/hipinhththu/pannuke"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

assert os.path.exists(CHECKPOINT_PATH), "Attach hipinhththu/sam3-native-pt"
assert os.path.exists(DATA_ROOT), "Attach hipinhththu/pannuke"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scikit-learn", "matplotlib", "opencv-python",
                "pycocotools", "einops", "tqdm"], check=True)
print("OK setup")

## Helper modules

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile sam3_train.py
from __future__ import annotations
from typing import Dict, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

def make_transform(resolution: int = 1008):
    return v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(resolution, resolution)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

@torch.no_grad()
def encode_image_frozen(model, transform, pil_img, device: str = "cuda"):
    image = v2.functional.to_image(pil_img).to(device)
    image = transform(image).unsqueeze(0)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        backbone_out = model.backbone.forward_image(image)

        
        inst_pred = getattr(model, "inst_interactive_predictor", None)
        if inst_pred is not None and "sam2_backbone_out" in backbone_out:
            sam2_bb = backbone_out["sam2_backbone_out"]
            sam2_bb["backbone_fpn"][0] = (
                inst_pred.model.sam_mask_decoder.conv_s0(sam2_bb["backbone_fpn"][0])
            )
            sam2_bb["backbone_fpn"][1] = (
                inst_pred.model.sam_mask_decoder.conv_s1(sam2_bb["backbone_fpn"][1])
            )
    return backbone_out

def encode_text(model, prompt: str, device: str = "cuda"):
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            return model.backbone.forward_text([prompt], device=device)

def forward_decoder_with_grad(model, backbone_out: Dict, find_stage,
                              geometric_prompt, device: str = "cuda") -> Dict:
    was_training = model.training
    if was_training:
        model.eval()
    try:
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            outputs = model.forward_grounding(
                backbone_out=backbone_out,
                find_input=find_stage,
                geometric_prompt=geometric_prompt,
                find_target=None,
            )
    finally:
        if was_training:
            model.train()
    return outputs

def semantic_union_mask(outputs: Dict, target_hw: Tuple[int, int]) -> torch.Tensor:
    
    pred_logits = outputs["pred_logits"].float()         
    pred_masks  = outputs["pred_masks"].float()          
    pres_logit  = outputs["presence_logit_dec"].float()  

    
    cls_prob = pred_logits.sigmoid()                   
    pres     = pres_logit.sigmoid().unsqueeze(1)       
    prob     = (cls_prob * pres).squeeze(-1)           

    
    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid()  

    
    weighted = prob[:, :, None, None] * masks   

    
    
    
    eps = 1e-4
    log_one_minus = torch.log(torch.clamp(1.0 - weighted, min=eps, max=1.0 - eps))
    log_prod = log_one_minus.sum(dim=1)         
    union = 1.0 - torch.exp(torch.clamp(log_prod, min=-20.0))  
    union = torch.clamp(union, min=eps, max=1.0 - eps)  
    return union.squeeze(0)                     

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def bce_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    p = torch.clamp(pred, eps, 1 - eps)
    return -(target * torch.log(p) + (1 - target) * torch.log(1 - p)).mean()

def semantic_seg_loss(pred: torch.Tensor, target: torch.Tensor,
                      bce_weight: float = 0.5,
                      dice_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    total = bce_weight * bce + dice_weight * dice
    return total, {"bce": float(bce.item()), "dice": float(dice.item()),
                   "loss": float(total.item())}

@torch.no_grad()
def inference_to_binary(outputs: Dict, target_hw: Tuple[int, int],
                        score_threshold: float = 0.3) -> torch.Tensor:
    pred_logits = outputs["pred_logits"]
    pred_masks  = outputs["pred_masks"]
    pres_logit  = outputs["presence_logit_dec"]

    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)  

    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)  

    keep = prob > score_threshold
    if keep.sum() == 0:
        return torch.zeros(target_hw, dtype=torch.bool, device=pred_logits.device)

    selected = (masks[keep] > 0.5)
    union = selected.any(dim=0)
    return union

In [ ]:
import sys
for p in [".", "/kaggle/working", SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
from PIL import Image
from tqdm import tqdm

from pannuke_loader import PanNukeFold, CELL_TYPES, DEFAULT_ROOT
from metrics import ClassWiseAccumulator, PerPromptClassAccumulator
from sam3_train import (make_transform, encode_image_frozen, encode_text,
                        forward_decoder_with_grad, inference_to_binary)
print("Helpers loaded.")

## 01 — Build SAM3 zero-shot (NO LoRA)

In [ ]:
from sam3.model_builder import build_sam3_image_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam3_image_model(
    device=device, eval_mode=True,
    checkpoint_path=CHECKPOINT_PATH, load_from_HF=False,
)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"SAM3 params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("Zero-shot baseline ready.")

## 02 — Kong-protocol matched subset (~33% of Fold 3)

In [ ]:
KONG_SUBSET_RATIO = 3/9
SEED = 42

fold3 = PanNukeFold(DEFAULT_ROOT, 3)
np.random.seed(SEED)
n_subset = int(len(fold3) * KONG_SUBSET_RATIO)
subset_indices = np.random.choice(len(fold3), n_subset, replace=False)

print(f"Fold 3 total: {len(fold3)}")
print(f"Kong-subset (33%): {n_subset} patches  (seed={SEED})")

## 03 — Prompt strategies (paper protocol)

In [ ]:
PROMPTS_MEDICAL = {
    "Neoplastic":   ["histopathology image of neoplastic tissue"],
    "Inflammatory": ["histopathology image of inflammatory tissue"],
    "Connective":   ["histopathology image of connective tissue"],
    "Dead":         ["histopathology image of dead tissue"],
    "Epithelial":   ["histopathology image of epithelial tissue"],
}
PROMPTS_LLM = {
    "Neoplastic":   ["Neoplastic cell", "Tumor cell", "Cancer cell", "Malignant cell"],
    "Inflammatory": ["Inflammatory cell", "Lymphocyte", "Immune cell", "Leukocyte"],
    "Connective":   ["Connective tissue cell", "Fibroblast", "Stromal cell"],
    "Dead":         ["Dead cell", "Apoptotic cell", "Necrotic cell"],
    "Epithelial":   ["Epithelial cell", "Epithelium", "Lining cell",
                     "Surface cell", "Mucosal cell nucleus"],
}
PROMPT_GENERIC = "cell"
SCORE_THRESH = 0.3
print("Prompts ready.")

## 04 — Inference helpers

In [ ]:
from sam3.model.data_misc import FindStage

transform = make_transform(resolution=1008)
find_stage = FindStage(
    img_ids=torch.tensor([0], device=device, dtype=torch.long),
    text_ids=torch.tensor([0], device=device, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

@torch.no_grad()
def encode_cached(pil): return encode_image_frozen(model, transform, pil, device=device)

@torch.no_grad()
def predict(state, prompt):
    st = dict(state)
    text_out = encode_text(model, prompt, device=device)
    st.update(text_out)
    outputs = forward_decoder_with_grad(model, st, find_stage, model._get_dummy_prompt())
    pm = inference_to_binary(outputs, target_hw=(256, 256), score_threshold=SCORE_THRESH)
    return pm.cpu().numpy().astype(bool)

## 05 — Eval on Kong subset

In [ ]:
acc_med = ClassWiseAccumulator(CELL_TYPES)
acc_llm = PerPromptClassAccumulator(CELL_TYPES, PROMPTS_LLM)
acc_gen = ClassWiseAccumulator(CELL_TYPES)

t0 = time.time()
for idx in tqdm(subset_indices, desc="Kong subset eval"):
    sample = fold3[int(idx)]
    pil = Image.fromarray(sample["image"]).convert("RGB")
    gt = {c: (sample["masks"][CELL_TYPES.index(c)] > 0) for c in CELL_TYPES}
    state = encode_cached(pil)

    pred_gen = predict(state, PROMPT_GENERIC)
    for c in CELL_TYPES: acc_gen.update(pred_gen, gt[c], c)
    for c in CELL_TYPES:
        pred_m = predict(state, PROMPTS_MEDICAL[c][0])
        acc_med.update(pred_m, gt[c], c)
    for c, prompts in PROMPTS_LLM.items():
        for p in prompts:
            pred_l = predict(state, p)
            acc_llm.update(pred_l, gt[c], c, p)

elapsed = time.time() - t0
print(f"\nDone. {elapsed/60:.1f}min ({elapsed/n_subset:.1f}s/image)")

## 06 — Compare vs Kong Table 1

In [ ]:
KONG_TABLE1 = {
    "Medical": {"mIoU": 0.26, "Dice": 0.37},
    "LLM":     {"mIoU": 4.08, "Dice": 5.16},
    "Generic": {"mIoU": 6.22, "Dice": 8.13},
}

results = {
    "Medical": acc_med.summary(),
    "LLM":     acc_llm.summary(),
    "Generic": acc_gen.summary(),
}

print("=" * 90)
print(f"PHASE A1 — Kong-protocol subset (N={n_subset}, seed={SEED})")
print("=" * 90)
print(f"\n{'Strategy':12s} | {'Kong paper':>12s} | {'A1 Kong-subset':>18s} | {'Δ':>8s}")
print("-" * 90)
for name, kong in KONG_TABLE1.items():
    ours = results[name]["mIoU"] * 100
    delta = ours - kong["mIoU"]
    sign = "+" if delta >= 0 else ""
    print(f"{name:12s} | {kong['mIoU']:>11.2f}% | {ours:>17.2f}% | {sign}{delta:>6.2f}pp")

print("\nPer-class mIoU (Kong-subset):")
print(f"{'Class':14s} | {'Medical':>9s} | {'LLM-avg':>9s} | {'Generic':>9s}")
print("-" * 60)
for c in CELL_TYPES:
    m = results["Medical"]["per_class"][c]["IoU"] * 100
    l = results["LLM"]["per_class"][c]["IoU"] * 100
    g = results["Generic"]["per_class"][c]["IoU"] * 100
    print(f"  {c:12s} | {m:>8.2f}% | {l:>8.2f}% | {g:>8.2f}%")

In [ ]:
out_path = f"{WORK}/phase_A1_kong_subset_results.json"
with open(out_path, "w") as f:
    json.dump({
        "config": {"n_subset": int(n_subset), "seed": SEED,
                    "ratio": KONG_SUBSET_RATIO, "elapsed_minutes": elapsed/60},
        "kong_reference": KONG_TABLE1,
        "ours_kong_subset": results,
    }, f, indent=2)
print(f"Saved: {out_path}")
print("\nUse for paper Appendix Table A1 — direct Kong comparison.")

## Notes for paper Section 4.1

Declare protocol khác nhau:

```
Main results (Table 1): Gamper standard PanNuke protocol
  Train Folds 1+2 (N=5,181), test full Fold 3 (N=2,722).

Appendix (Table A1): Kong-protocol matched subset
  Random 33% subset of Fold 3 (N=907, seed=42),
  matching Kong et al. 2025 internal 5:1:3 split for direct numerical compare.
```